In [ ]:
import paulssonlab.deaton.trenchripper.trenchripper as tr

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from IPython.display import Video
# import warnings
# warnings.filterwarnings(action='once',category=UserWarning)

# import holoviews as hv
# hv.extension('bokeh')

import h5py
import matplotlib.pyplot as plt
import numpy as np
import skimage as sk
import dask.dataframe as dd

In [ ]:
dask_controller = tr.trcluster.dask_controller(
    walltime="02:00:00",
    local=False,
    n_workers=50,
    n_workers_min=50,
    memory="4GB",
    working_directory="/home/lag36/scratch/lag36/dask/"
)
dask_controller.startdask()

#small testing deployment

# dask_controller = tr.trcluster.dask_controller(
#     walltime="02:00:00",
#     local=False,
#     n_workers=40,
#     n_workers_min=40,
#     memory="16GB",
#     working_directory="/home/de64/scratch/de64/dask",
# )
# dask_controller.startdask()

In [ ]:
dask_controller.shutdown()

In [ ]:
dask_controller.daskclient

In [ ]:
# Identify trenches with dnaA from tables. I need the 
# Pull kymo image
bdf = dd.read_parquet("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Final_Barcodes_df_Merged", engine="pyarrow",calculate_divisions=True).persist()

In [ ]:
trench_list = sorted(bdf[bdf['oDEPool7_id'] == 1422]
           .groupby(['File Index', 'File Trench Index']).apply(lambda x:x.iloc[0]).index.compute())

In [ ]:
trench_list_ftsA = trench_list

In [ ]:
trench_list_ftsY = trench_list

In [ ]:
trench_list_rpsB = trench_list

In [ ]:
trench_list_infC = trench_list

In [ ]:
def load_trench_lists(oDE_ids, gene_names):
    trench_lists = {}
    for i, oDE_id in enumerate(oDE_ids):
        trench_lists[gene_names[i]] = sorted(bdf[bdf['oDEPool7_id'] == oDE_id]
           .groupby(['File Index', 'File Trench Index']).apply(lambda x:x.iloc[0]).index.compute())
    return trench_lists

In [ ]:
trench_list = sorted(bdf[bdf['oDEPool7_id'] == 1422]
           .groupby(['File Index', 'File Trench Index']).apply(lambda x:x.iloc[0]).index.compute())
oDE_ids = [27394, 25930, 25917, 23531, 8674, 28219, 1829, 1846,1422]
gene_names = ['ftsA', 'ftsY_3', 'ftsY_1', 'infC_3', 'infC_10', 'mukE', 'nusG_8', 'nusG_3', 'secE']

In [ ]:
trench_list = sorted(bdf[bdf['oDEPool7_id'] == 1422]
           .groupby(['File Index', 'File Trench Index']).apply(lambda x:x.iloc[0]).index.compute())
oDE_ids = [27394, 25930, 25917, 23531, 8674, 28219, 1829, 1846,1422]
gene_names = ['ftsA', 'ftsY_3', 'ftsY_1', 'infC_3', 'infC_10', 'mukE', 'nusG_8', 'nusG_3', 'secE']
plot_all_kymos(trench_list, 0)

In [ ]:
trench_lists = load_trench_lists(oDE_ids, gene_names)

In [ ]:
def plot_all_kymos(trench_list, idx):
    file_idx = trench_list[idx][0]
    file_tr_idx = trench_list[idx][1]
    indices = list(range(100,200,2))
    # YFP seg     
    with h5py.File('/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/Growth_Division/fluorsegmentation/segmentation_'+str(file_idx)+'.hdf5', 'r') as infile:
        img = infile['data'][file_tr_idx][indices[0]]
        plt.figure(figsize=(12,12)) 
        for ind in indices:
            img_ind = infile['data'][file_tr_idx][ind]
            img = np.concatenate((img, img_ind), axis = 1)
        plt.imshow(img)
        plt.xticks([])
        plt.show()   
    channels = ['YFP', 'mCherry']
    with h5py.File('/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/Growth_Division/kymograph/kymograph_'+str(file_idx)+'.hdf5', 'r') as infile:
        for ch in channels:
            plt.figure(figsize=(12,12)) 
            img = infile[ch][file_tr_idx][indices[0]]
            for ind in indices:
                img_ind = infile[ch][file_tr_idx][ind]
                img = np.concatenate((img, img_ind), axis = 1)
            plt.imshow(img, cmap='gray')
            plt.xticks([])
            plt.show()
    #mCherry seg    
    with h5py.File('/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/Growth_Division/nucleoidsegmentation/segmentation_'+str(file_idx)+'.hdf5', 'r') as infile:
        img = infile['data'][file_tr_idx][indices[0]]
        plt.figure(figsize=(12,12)) 
        for ind in indices:
            img_ind = infile['data'][file_tr_idx][ind]
            img = np.concatenate((img, img_ind), axis = 1)
        plt.imshow(img)
        plt.xticks([])
        plt.show()

In [ ]:
trench_list_ftsA

In [ ]:
plot_all_kymos(trench_list, 0)

## Function to apply Hessian (roughness of nucleoid)

In [ ]:
trench_list_infC

In [ ]:
dask_controller.shutdown()

In [ ]:
def load_images_for_testing(trench_list, idx, t_idx, show_imgs=False):

    file_idx = trench_list[idx][0]
    file_tr_idx = trench_list[idx][1]
    
    imgs_fl = []
    channels = ['YFP', 'mCherry']
    with h5py.File('/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/Growth_Division/kymograph/kymograph_'+str(file_idx)+'.hdf5', 'r') as infile:
        for ch in channels:
            img = infile[ch][file_tr_idx][t_idx]
            imgs_fl.append(img)
            if show_imgs:
                plt.figure(figsize=(12,12)) 
                plt.imshow(img, cmap='gray')
                plt.xticks([])
                plt.show()

    imgs_mask = []

    with h5py.File('/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/Growth_Division/fluorsegmentation/segmentation_'+str(file_idx)+'.hdf5', 'r') as infile:
        img = infile['data'][file_tr_idx][t_idx] 
        imgs_mask.append(img)
        if show_imgs:
            plt.figure(figsize=(12,12))
            plt.imshow(img)
            plt.xticks([])
            plt.show()

    with h5py.File('/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/Growth_Division/nucleoidsegmentation/segmentation_'+str(file_idx)+'.hdf5', 'r') as infile:
        img = infile['data'][file_tr_idx][t_idx] 
        imgs_mask.append(img)
        if show_imgs:
            plt.figure(figsize=(12,12))
            plt.imshow(img)
            plt.xticks([])
            plt.show()
    return imgs_fl, imgs_mask

In [ ]:
idx = 3
t_idx = 180
show_imgs = False

imgs_fl, imgs_mask = load_images_for_testing(trench_list_infC, 3, 180, False)
plt.imshow(imgs_fl[1], cmap='gray')
plt.show()
plt.close()
mean_hessian(imgs_mask[1], imgs_fl[1])

In [ ]:
idx = 3
t_idx = 180
show_imgs = False

imgs_fl, imgs_mask = load_images_for_testing(trench_list_infC, 3, 180, False)
plt.imshow(imgs_fl[0], cmap='gray')
plt.show()
plt.close()
mean_hessian(imgs_mask[0], imgs_fl[0])

In [ ]:
trench_list_rpsB

In [ ]:
#function must take mask_arr and may optionally take img_arr (by Daniel)
def mean_hessian(mask_arr,img_arr,edge_padding=6, show_plots = False):
    padded = np.pad(img_arr, edge_padding, 'edge')
    hessian = sk.feature.hessian_matrix(padded,order="rc")
    eig_img = sk.feature.hessian_matrix_eigvals(hessian)
    eig_img = np.max(eig_img,axis=0)
    eig_img = eig_img[edge_padding:-edge_padding,edge_padding:-edge_padding]
    if show_plots:
        plt.imshow(eig_img)
    eig_vals = eig_img[mask_arr]
    mean_eig_val = np.mean(eig_vals)
    return mean_eig_val

In [ ]:
#function must take mask_arr and may optionally take img_arr (by Daniel)
def std(mask_arr,img_arr,edge_padding=6, show_plots = False):
    eig_vals = img_arr[mask_arr]
    mean_eig_val = np.std(eig_vals)/np.mean(eig_vals)
    return mean_eig_val

In [ ]:
#function must take mask_arr and may optionally take img_arr (by Daniel)
def skewness(mask_arr,img_arr,edge_padding=6, show_plots = False):
    eig_vals = img_arr[mask_arr]
    moments = sk.measure.moments_normalized(eig_vals, 4)[
    mean_eig_val = 
    return mean_eig_val

In [ ]:
def get_result_list_single_frame(trench_list,
                                 idx,
                                 t_idx,
                                 ch_idx_imgs,
                                 ch_idx_masks,
                                 show_imgs=False):
    imgs_fl, imgs_mask = load_images_for_testing(trench_list=trench_list, 
                                                 idx = idx, 
                                                 t_idx = t_idx,
                                                 show_imgs = show_imgs)
    
    img_arr = imgs_fl[ch_idx_imgs]
    mask_arr_all = imgs_mask[ch_idx_masks]
    
    n_segments = mask_arr_all.max() # Get number of segment on current image
    # Run over all segments
    list_results = [0]*n_segments
    for i in range(1, n_segments+1):
        mask_arr = np.where(mask_arr_all == i, True, False)
        # plt.imshow(mask_arr)
        # plt.show()
        # plt.close()
        list_results[i-1] = mean_hessian(mask_arr=mask_arr,
                                         img_arr=img_arr)
    return list_results

In [ ]:
idx = 0
t_idx = 180
# ch_idx = 1 # 0: YFP # 1: mCherry
ch_idx_imgs = 1
ch_idx_masks = 0
show_imgs = False

get_result_list_single_frame(trench_list = trench_list_infC,
                             idx = idx,
                             t_idx = t_idx,
                             ch_idx_imgs = ch_idx_imgs,
                             ch_idx_masks = ch_idx_masks,
                             show_imgs=False)

In [ ]:
def get_result_list_all_frames(trench_list,
                               idx,
                               t_i,
                               t_f,
                               ch_idx_imgs,
                               ch_idx_masks,
                               show_imgs = False):
    list_results_all_t = []
    for t_idx in range(t_i, t_f):
        list_results_all_t += get_result_list_single_frame(trench_list = trench_list,
                                           idx = idx,
                                           t_idx = t_idx,
                                           ch_idx_imgs = ch_idx_imgs,
                                           ch_idx_masks = ch_idx_masks,
                                           show_imgs=False)
    return list_results_all_t

In [ ]:
trench_list = trench_list_rpsB
idx = 0
t_i = 160
t_f = 200
ch_idx_imgs = 1
ch_idx_masks = 0

list_rpsB_nucs = get_result_list_all_frames(trench_list,
                                            idx,
                                            t_i,
                                            t_f,
                                            ch_idx_imgs,
                                            ch_idx_masks,
                                            show_imgs = False)

In [ ]:
def get_result_list_all_trenches(trench_list,
                                 t_i,
                                 t_f,
                                 ch_idx_imgs,
                                 ch_idx_masks,
                                 show_imgs=False):
    list_results_all_trenches = []
    for idx in range(len(trench_list)):
        print( idx, '/', len(trench_list))
        list_results_all_trenches += get_result_list_all_frames(trench_list,
                                                                idx,
                                                                t_i,
                                                                t_f,
                                                                ch_idx_imgs,
                                                                ch_idx_masks,
                                                                show_imgs = False)
    return list_results_all_trenches

In [ ]:
trench_list = trench_list_rpsB
t_i = 160
t_f = 200
ch_idx_imgs = 1
ch_idx_masks = 0
list_results_rpsB_nuc = get_result_list_all_trenches(trench_list,
                                                 t_i,
                                                 t_f,
                                                 ch_idx_imgs,
                                                 ch_idx_masks,
                                                 show_imgs=False)

trench_list = trench_list_infC
t_i = 160
t_f = 200
ch_idx_imgs = 1
ch_idx_masks = 0
list_results_infC_nuc = get_result_list_all_trenches(trench_list,
                                                 t_i,
                                                 t_f,
                                                 ch_idx_imgs,
                                                 ch_idx_masks,
                                                 show_imgs=False)

In [ ]:
idx = 0
t_i = 160
t_f = 200
ch_idx_imgs = 1
ch_idx_masks = 1
metric_lists_mean_hess = {}
for gene_name in trench_lists:
    trench_list = trench_lists[gene_name]
    metric_lists_mean_hess[gene_name] = get_result_list_all_frames(trench_list,
                                                         idx,
                                                         t_i,
                                                         t_f,
                                                         ch_idx_imgs,
                                                         ch_idx_masks,
                                                         show_imgs = False)

In [ ]:
lo = 5
hi = 10
for i, gene_name in enumerate(metric_lists_mean_hess):
    if i >= lo and i < hi:
        plt.hist(metric_lists_mean_hess[gene_name], label = gene_name, alpha = 0.5, density=True)
   
    
    plt.legend()

In [ ]:
lo = 5
hi = 10
for i, gene_name in enumerate(metric_lists):
    if i >= lo and i < hi:
        plt.hist(metric_lists_cv[gene_name], label = gene_name, alpha = 0.5, density=True)
   
    
    plt.legend()

In [ ]:
lo = 5
hi = 10
for i, gene_name in enumerate(metric_lists):
    if i >= lo and i < hi:
        plt.hist(metric_lists_cv[gene_name], label = gene_name, alpha = 0.5, density=True)
   
    
    plt.legend()

In [ ]:
lo = 1
hi = 6
for i, gene_name in enumerate(metric_lists):
    if i >= lo and i < hi:
        plt.hist(metric_lists_std[gene_name], label = gene_name, alpha = 0.5, density=True)
   
    
    plt.legend()

In [ ]:
lo = -1
hi = 4
for i, gene_name in enumerate(metric_lists):
    if i > lo and i < hi:
        plt.hist(metric_lists[gene_name], label = gene_name, alpha = 0.5, density=True)
   
    
    plt.legend()

In [ ]:
oDE_ids =    [27394 , 25930   , 25917   ,  23531  ,  8674    , 28219 , 1829    , 1846    , 1422]
gene_names = ['ftsA', 'ftsY_3', 'ftsY_1', 'infC_3', 'infC_10', 'mukE', 'nusG_8', 'nusG_3', 'secE']

In [ ]:
trench_list = sorted(bdf[bdf['oDEPool7_id'] == 25917]
           .groupby(['File Index', 'File Trench Index']).apply(lambda x:x.iloc[0]).index.compute())

plot_all_kymos(trench_list, 0)

In [ ]:
trench_list = trench_list_ftsA
idx = 0
t_i = 160
t_f = 200
ch_idx_imgs = 1
ch_idx_masks = 0

list_ftsA_nucs = get_result_list_all_frames(trench_list,
                                            idx,
                                            t_i,
                                            t_f,
                                            ch_idx_imgs,
                                            ch_idx_masks,
                                            show_imgs = False)

trench_list = trench_list_ftsY
idx = 0
t_i = 160
t_f = 200
ch_idx_imgs = 1
ch_idx_masks = 0

list_ftsY_nucs = get_result_list_all_frames(trench_list,
                                            idx,
                                            t_i,
                                            t_f,
                                            ch_idx_imgs,
                                            ch_idx_masks,
                                            show_imgs = False)

In [ ]:
rpsB_list_results_all_t_cells = list_results_all_t

In [ ]:
rpsB_list_results_all_t_nuc = list_results_all_t

In [ ]:
list_results_all_t_cells = list_results_all_t

In [ ]:
list_results_all_t_nuc = list_results_all_t

In [ ]:
plt.hist(list_results_infC_nuc, bins=20, alpha = 0.5)
plt.hist(list_results_rpsB_nuc, bins=20, alpha = 0.5)

In [ ]:
plt.hist(list_results_all_t_cells, label = 'infC cyto', alpha = 0.5)
plt.hist(list_results_all_t_nuc, label = 'infC nucs', alpha = 0.5)

plt.hist(rpsB_list_results_all_t_cells, label = 'rpsB cyto',alpha = 0.5)
plt.hist(rpsB_list_results_all_t_nuc, label = 'rpsB nucs', alpha = 0.5)

# plt.hist(list_rpsB_nucs)

plt.legend()

In [ ]:
idx = 3
t_idx = 180
show_imgs = False

imgs_fl, imgs_mask = load_images_for_testing(trench_list_rpsB, 3, 180, False)
plt.imshow(imgs_fl[1], cmap='gray')
plt.show()
plt.close()
mean_hessian(imgs_mask[1], imgs_fl[1])

In [ ]:
mask_arr = imgs_mask[0]
img_arr = imgs_fl[0]
mean_hessian(mask_arr, img_arr)

In [ ]:
mask_arr = imgs_mask[1]
img_arr = imgs_fl[1]
mean_hessian(mask_arr, img_arr)

In [ ]:
mask_arr = imgs_mask[1]
img_arr = imgs_fl[1]
mean_hessian(mask_arr, img_arr)

In [ ]:
import pickle
headpath = '/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/Growth_Division/'
with open(headpath+'fluorescent_segmentation.par', 'rb') as pfile:
    a = pickle.load(pfile)
a